# Chapter 03 — N-gram MLP Language Model

The bigram model only looked at one character of context. But language has structure that spans much further — the word 'transformer' only makes sense given several preceding characters. N-gram models extend our context window, and the Multi-Layer Perceptron (MLP) gives us a neural network approach to learning these longer patterns.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NgramMLP(nn.Module):
    def __init__(self, vocab_size, context_len, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(context_len * embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x: (batch, context_len) of token indices
        emb = self.embedding(x)          # (batch, ctx, embed)
        emb = emb.view(emb.size(0), -1)  # (batch, ctx * embed)
        h = F.gelu(self.fc1(emb))        # (batch, hidden)
        logits = self.fc2(h)             # (batch, vocab_size)
        return logits

# Demo: forward pass with synthetic data
model = NgramMLP(vocab_size=65, context_len=8, embed_dim=32, hidden_dim=128)
x_batch = torch.randint(0, 65, (4, 8))   # 4 samples, context_len=8
y_batch = torch.randint(0, 65, (4,))      # target tokens

logits = model(x_batch)
loss = F.cross_entropy(logits, y_batch)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input shape:  {tuple(x_batch.shape)}")
print(f"Output shape: {tuple(logits.shape)}")
print(f"Initial loss: {loss.item():.4f} (random ≈ {torch.log(torch.tensor(65.0)).item():.4f})")

---

**Course:** Aprenda LLM | **Chapter 03**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.